[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/EelcoHoogendoorn/numga/blob/main/examples/relativity/closed_universe/waves.ipynb)

# Gravitational Waves of a Closed Universe

This example computes the gravitational waves of a closed universe, the round three-sphere, on the 600-cell's 600 regular tetrahedra, and watches them from inside through three small crystals. Regge calculus holds a curved space in the lengths of a mesh's edges: each tetrahedron is a piece of the space with those lengths, and the curvature gathers at the edges, as the deficit by which the angles of the tetrahedra round an edge fall short of a whole turn. A gravitational wave changes the lengths, and the deficits it causes pull them back. The slowest wave of the closed universe, its squash, stretches the whole sphere along one direction and squeezes it along another, and swings back. Three crystals of atoms on springs feel it differently: a soft one goes along with space, a stiff one keeps its shape, and one tuned to the swing rings with it.

Points of the sphere are antivectors, and a tetrahedron's ramps, the planes through its faces scaled to read one at the opposite corners, read a point's barycentric weights, `ramps & point`. With the squared lengths, the ramps give the metric of the cone over the tetrahedron from the sphere's centre, a map from points to planes, and its inverse pairs two ramps to the cosine of the angle between their faces: each tetrahedron is read as a piece of the round sphere, so that the round sphere's own chords leave no deficit on any mesh. The waves are the changes of the squared lengths that no motion of the vertices makes, kept so by one solve over the vertices. Stepped through time they are a smooth field of strain, read in the isoclinic frames: at every place, a map from a step to the plane that reads its stretch, through which the crystals' atoms and the light roll.

In the notation of Regge calculus, the action of a curved three-space is $S = \sum_e \ell_e\,\epsilon_e$ over its edges' lengths and deficit angles, which by Schläfli's identity changes as $\delta S = \sum_e \epsilon_e\,\delta\ell_e$, and the linear waves of the squared lengths $q$ solve $M\ddot q = -K q$, with $K$ the Hessian of the potential and $M$ the discrete DeWitt metric, restricted to the transverse traceless changes.

In [ ]:
# In Colab, numga and its examples, from GitHub.
import sys

if "google.colab" in sys.modules:
    %pip install -q "numga[examples] @ git+https://github.com/EelcoHoogendoorn/numga"

In [ ]:
%matplotlib inline
from collections.abc import Iterator
from functools import reduce
from operator import and_

import numpy as np
from numga import stack
from numga.sparse import spdiag

from examples import instantiate
from examples.util.animation import show_animation
from examples.relativity.closed_universe import core, render, scenarios

np.set_printoptions(precision=4, suppress=True)

ga = core.ga
mv = instantiate("examples.util.mesh", ga).mv
Scalar = ga.gatype.scalar()                                                     # Scalar+
# Planes: the ramps across the tetrahedra, and those through the origin, the polar planes of its steps.
Plane = ga.gatype.vector()                                                      # Vector+
Through = Plane.derive.within("x y z")                                          # Vector+
# Points of the sphere and the origin's steps; the motors that carry them.
Point = ga.gatype.antivector()                                                  # Antivector-
Step = Point.derive.containing("w")                                             # Antivector-
Motor = ga.gatype.rotor()                                                       # Rotor+
Translating = ga.gatype((Step, Through))                                        # Antivector- <- Vector+

# The origin's equator, the plane `w`, and the origin, its pole; the extrinsic pseudoscalar.
equator = mv.w                                                                  # [] Plane+
origin = equator.dual()                                                         # [] Point-
pseudoscalar = mv.xyzw                                                          # [] Antiscalar+


def isoclinic(places: Point) -> Motor:
    """The isoclinic frames at places on the sphere, `Motor+`: the motors that carry the origin to each by turning
    alike in the plane of the great circle between them and in the plane square to it."""
    # The half of the quotient of the place by the origin that the pseudoscalar keeps, with one in the other half.
    return ((1 + pseudoscalar) * (places.normalized() / origin) + (1 - pseudoscalar)) / 2

## The 600-cell at rest

The 600-cell's vertices are the 120 unit icosians, placed by a turn, and its edges' squared lengths at rest are those of the round sphere's chords between them. Read as a piece of the round sphere, a tetrahedron is the cone over it from the sphere's centre: the hyperplane through its corners takes the centre, and each pair of ramps reads the squared chord between their corners, so that the squared lengths make the cone's metric. Through the inverse metric, the ramps of the two corners off an edge pair to the cosine of the angle between the faces that meet there, and a whole turn less those angles round the edge is its deficit. The deficits drive the lengths: the potential's slope along each squared length is the deficit, scaled by how the arc changes with the squared chord.

In [ ]:
cell_mesh, cell_vertices = scenarios.six_hundred_cell()
chords = cell_mesh.d[0] * cell_vertices                                         # Point-[edges]
rest = chords.dot(chords)                                                       # Scalar+[edges]
corners = cell_mesh.incidence[3, 0] * cell_vertices                             # [corners] Point-[cells]
# Each corner's ramp: the plane through the face across from it, scaled to read one at the corner.
faces = reduce(and_, corners[cell_mesh.corners[3, 2]])                          # [corners] Antivector-[cells]
ramps = faces / (faces & corners)                                               # [corners] Plane+[cells]
# The hyperplane through the corners, one on each, and each pair of ramps' symmetric product, which reads the squared
# chord between their corners: weighted by the squared lengths, the cone's metric from the centre.
radial = ramps.sum(axis=0)                                                      # Plane+[cells]
tails, heads = cell_mesh.corners[3, 1]
cone_basis = -(ramps[tails] * (ramps[heads] & Point) + ramps[heads] * (ramps[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
cone_map = (spdiag(cone_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]


def deficits(squared_lengths: Scalar) -> Scalar:
    """The deficit angle at each edge, `Scalar+[edges]`, of the tetrahedra read as pieces of the round sphere with
    these squared lengths: a whole turn less the angles between the faces round the edge."""
    cone = radial * (radial & Point) + cone_map * squared_lengths               # (Plane+ <- Point-)[cells]
    pairings = ramps[:, None] & cone.batch().inverse().field()(ramps[None, :])   # [corners, corners] Scalar+[cells]
    # The faces at an edge are across from the two corners off it, the opposite edge's: in the tetrahedron's own order
    # of its edges, the order reversed.
    off_edges = pairings[tails[::-1], heads[::-1]]                              # [cell_edges] Scalar+[cells]
    sides = pairings[tails[::-1], tails[::-1]] * pairings[heads[::-1], heads[::-1]]   # [cell_edges] Scalar+[cells]
    angles = (-off_edges / sides.square_root()).arccos()                        # [cell_edges] Scalar+[cells]
    return 2 * np.pi - (~cell_mesh.incidence[3, 1] * angles).sum(axis=-1)


def slopes(squared_lengths: Scalar) -> Scalar:
    """The potential's slope along each squared length, `Scalar+[edges]`: the deficit, times how the arc changes with
    the squared chord, a half over the chord and the cosine of half the arc."""
    return -deficits(squared_lengths) / (squared_lengths * (1 - squared_lengths / 4)).square_root()

## Masses and the waves

The lengths' motion has a kinetic term of its own, relativity's on a flat tetrahedron: a quarter of its volume times the pairings, through its inverse metric, of the corners across two edges, both ways. The flat tetrahedron's metric comes from the same products, of its corner gradients, the ramps less a quarter of the hyperplane. The stiffness is the slopes' change with the lengths, found by nudging them. A move of the vertices changes each edge's squared length by twice the chord's reading of the move; such changes are no waves, and the waves are what is left once the change nearest by the masses that the moves make is taken away, a solve over the vertices. This cell takes a few seconds, most of it finding the stiffness.

In [ ]:
# The flat tetrahedra: the corner gradients, their products, and each one's metric and volume from the squared lengths.
gradients = ramps - radial / 4                                                  # [corners] Plane+[cells]
flat_basis = -(gradients[tails] * (gradients[heads] & Point) + gradients[heads] * (gradients[tails] & Point)) / 2   # [cell_edges] (Plane+ <- Point-)[cells]
flat_map = (spdiag(flat_basis) * cell_mesh.incidence[3, 1]).sum(axis=0)         # (Plane+ <- Point-)[cells <- edges]
flat = radial * (radial & Point) + flat_map * rest                              # (Plane+ <- Point-)[cells]
spans = stack((corners[1] - corners[0], corners[2] - corners[0], corners[3] - corners[0]))   # [3] Point-[cells]
volumes = ((flat(spans[0]) ^ flat(spans[1]) ^ flat(spans[2]) ^ radial) & (corners[0] & corners[1] & corners[2] & corners[3])).abs().square_root() / 6   # Scalar+[cells]
pairings = gradients[:, None] & flat.batch().inverse().field()(gradients[None, :])   # [corners, corners] Scalar+[cells]
# Relativity's kinetic term between the rates of two edges' squared lengths, assembled over the tetrahedra.
local = volumes / 4 * (pairings[tails[:, None], tails[None, :]] * pairings[heads[:, None], heads[None, :]]
                       + pairings[tails[:, None], heads[None, :]] * pairings[heads[:, None], tails[None, :]])   # [cell_edges, cell_edges] Scalar+[cells]
edges = cell_mesh.incidence[3, 1]                                               # [cell_edges] Scalar+[cells <- edges]
masses = (~edges[:, None] * spdiag(local) * edges[None, :]).sum(axis=0).sum(axis=0) * Scalar   # (Scalar+ <- Scalar+)[edges <- edges]
# The stiffness found by nudging each squared length by this much.
nudge = 1e-6
stiffness = core.jacobian(slopes, rest, nudge, cell_mesh)                       # (Scalar+ <- Scalar+)[edges <- edges]
moves = spdiag(2 * chords.dot(Point)) * cell_mesh.d[0]                          # (Scalar+ <- Point-)[edges <- vertices]
# The moves of the whole sphere change no length: this share of the largest move's own square settles them at nothing.
settling = 1e-10
reach = moves.adjoint()(masses(moves))                                          # (Point- <- Point-)[vertices <- vertices]
reach = reach + settling * core.largest(reach) * Point


def onto_waves(changes: Scalar) -> Scalar:
    """Changes of the squared lengths kept to the waves, `Scalar+[edges]`: less the change the vertex moves make
    that is nearest them by the masses."""
    return changes - moves(reach.solve(moves.adjoint()(masses(changes))))

## The squash

Read in the isoclinic frames, an edge runs along a step of the origin, and the squash lengthens the edges along `x` and shortens those along `y`, each by its step's share along the one less its share along the other, the same everywhere in the frames. Kept to the waves, its frequency squared is its stiffness over its masses: a fiftieth short of the continuum's eight, on edges a fifth of a half turn long.

In [ ]:
# Each edge's step, its head seen from its tail's isoclinic frame.
tail_points, head_points = cell_mesh.incidence[1, 0] * cell_vertices            # Point-[edges], Point-[edges]
steps = (isoclinic(tail_points).inverse() >> head_points).cast(Step)           # Antivector-[edges]
# The squash's change of the squared lengths, per unit swing of space: twice, as a squared length changes twice as
# fast as the length.
squash = 2 * rest * ((mv.x & steps).squared() - (mv.y & steps).squared()) / steps.dot(steps)   # Scalar+[edges]
pattern = onto_waves(squash)                                                    # Scalar+[edges]
squash_squared = float(((pattern * stiffness(pattern)).sites.sum() / (pattern * masses(pattern)).sites.sum()).to_array())
print(f"the squash's frequency squared: {squash_squared:.3f}")

## Stepped through time

The waves step by leapfrog, the stiffness's push turned into the rates' change by the masses and kept to the waves. Started from the squash at its swing, long before the first frame, so that the crystals settle into the swing and the light reaches the eye, they ring through one swing more. Stored at every step, they make a smooth field over the tetrahedra and over time, its strain blended from the vertices and read in the isoclinic frames, which the crystals and the light roll through. This cell takes a few seconds, most of it stepping.

In [ ]:
# The swing of space, the steps to a swing, and the swings the crystals settle over before the frames.
wave_swing = 0.12
n_steps_per_swing = 40
n_settling_swings = 8

In [ ]:
def waves_stepped(strains: Scalar, rates: Scalar, step: float) -> Iterator[tuple[Scalar, Scalar]]:
    """The squared lengths' changes and their rates, `Scalar+[edges]` each, at every step of a leapfrog under the
    stiffness and the masses, kept to the waves: half a kick, a drift, and the other half kick."""
    while True:
        yield strains, rates
        rates = rates - (step / 2) * onto_waves(masses.solve(stiffness(strains)))
        strains = strains + step * rates
        rates = rates - (step / 2) * onto_waves(masses.solve(stiffness(strains)))


frequency = np.sqrt(squash_squared)
period = 2 * np.pi / frequency
step = period / n_steps_per_swing
# From the settling swings and half a turn more, the light's time across the sphere, to the end of the frames' swing.
first = -(n_settling_swings + 1) * period - np.pi
n_levels = int(np.ceil((period - first) / step)) + 2
strains = wave_swing * np.cos(frequency * first) * pattern                     # Scalar+[edges]
rates = -wave_swing * frequency * np.sin(frequency * first) * pattern           # Scalar+[edges]
levels = [state for _, state in zip(range(n_levels), waves_stepped(strains, rates, step))]
waves = core.Waves(cell_mesh, cell_vertices, [changes for changes, _ in levels], [rates for _, rates in levels],
                   first, step)

## Three crystals

Three little tetrahedra of atoms, bonded along their edges by springs and dashpots, sit side by side ahead of the eye. The squash drives a tetrahedron's swing at root two times the root of its springs' stiffness: the tuned crystal's stiffness puts that swing at the waves' frequency, the soft one's is a sixteenth of it and the stiff one's sixteen times it, all damped to a quarter of critical. Each atom rolls through the waves along its great circle, its velocity turned and squeezed by them, while its bonds pull it along their proper separations; the soft crystal's bonds stretch with space, the stiff one's keep their length, and the tuned one's swing beyond the waves. This cell takes about ten seconds, most of it rolling the atoms.

In [ ]:
# The crystals' stiffness against the tuned one's, their damping as a share of critical, and the time step of their
# atoms.
tunings = np.array([1 / 16, 1.0, 16.0])                                         # [crystals]
critical_share = 0.25
crystal_step = 0.01
# The frames over one swing and a frame's duration in milliseconds; the eye's distance from the crystals, the time
# their light takes to arrive.
n_frames = 75
duration_ms = 40
crystal_distance = scenarios.crystal_distance

In [ ]:
bonds, atoms = scenarios.crystals(scenarios.crystals_ahead())
stiffnesses = frequency ** 2 / 2 * tunings                                      # [crystals]
dampings = critical_share * np.sqrt(2 * stiffnesses)                            # [crystals]
n_bonds_each = len(bonds.edges) // len(tunings)
bond_stiffness = core.mesh_module.as_scalar(np.repeat(stiffnesses, n_bonds_each)).field()   # Scalar+[bonds]
bond_damping = core.mesh_module.as_scalar(np.repeat(dampings, n_bonds_each)).field()        # Scalar+[bonds]
# A bond at rest is as long as its step, its head seen from its tail's frame.
bond_tails, bond_heads = bonds.incidence[1, 0] * atoms                          # Point-[bonds], Point-[bonds]
bond_steps = (isoclinic(bond_tails).inverse() >> bond_heads).cast(Step)        # Antivector-[bonds]
bond_rest = bond_steps.dot(bond_steps).square_root()                            # Scalar+[bonds]
eye_times = np.arange(n_frames) * period / n_frames                             # [frames]
# The light from the crystals left them their distance before it reached the eye.
at_rest = (mv.x.dual().cast(Step) * np.zeros(bonds.n_vertices)).field()         # Antivector-[atoms]
states = list(core.massive(cell_mesh, cell_vertices, waves, bonds, bond_rest, bond_stiffness, bond_damping, 0.0,
                           atoms, at_rest, eye_times[0] - crystal_distance - n_settling_swings * period, crystal_step,
                           eye_times - crystal_distance))

## The view from inside

The eye looks out through the waves at the crystals, the 600-cell's edges around it, its twelve guardrail decagons each in its own colour. Every ray rolls back through the waves from the eye, its heading turned and squeezed by the strain, so the struts sway as the squash swings; each crystal's bonds are coloured by their strain, blue squeezed and red stretched. This cell takes about forty seconds, most of it tracing the frames.

In [ ]:
guardrails = render.polygon_struts(scenarios.guardrails(cell_vertices, scenarios.guardrail_sides))
show_animation(render.inside(cell_mesh, cell_vertices, waves, scenarios.eye.broadcast_to(eye_times.shape), eye_times,
                             render.edge_styles(cell_mesh, *guardrails), render.wave_step,
                             render.crystal_frames(cell_mesh, cell_vertices, bonds, states, render.saturating_strain)),
               "closed_universe_600_cell", duration_ms)

In [ ]:
# checks
# The round sphere's chords leave no deficit; the squash is a fiftieth short of the continuum's eight; and over the
# swing the soft crystal carries the waves' strain, the tuned one rings at more than one and a half times it, and the
# stiff one keeps its shape.
assert np.abs(deficits(rest).to_array()).max() < 1e-12
assert 7.5 < squash_squared < 8
swings = np.abs(np.stack([strains.to_array() for _, strains in states])).reshape(len(states), len(tunings), -1)
soft, tuned, stiff = swings.max(axis=(0, 2))
assert 0.5 * wave_swing < soft < 1.5 * wave_swing and tuned > 1.5 * soft and stiff < 0.2 * wave_swing